In [14]:

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cv2
import seaborn as sns
from matplotlib.image import imread
from PIL import Image
import tensorflow as tf
np.random.seed(1337)
import gc
from tensorflow.keras.utils import to_categorical

import glob

from tensorflow.keras import layers
from keras.callbacks import ReduceLROnPlateau
from tensorflow.keras.models import Sequential,Model
from tensorflow.keras.layers import Input,Reshape,Multiply, Conv2DTranspose,Dropout, AveragePooling2D,Flatten, Dense, Conv2D,MaxPool2D, MaxPooling2D, BatchNormalization,concatenate,UpSampling2D
from tensorflow.keras.callbacks import EarlyStopping,ModelCheckpoint
import warnings
warnings.filterwarnings('ignore')


img_size = 128
dataset = os.listdir("music_dataset_spectro_full/train")
labels = dataset
print(labels)

['Accordion', 'Acoustic_Guitar', 'Banjo', 'Bass_Guitar', 'Clarinet', 'cowbell', 'Dobro', 'Drum_set', 'Electric_Guitar', 'flute', 'Harmonium', 'Horn', 'Keyboard', 'Mandolin', 'Organ', 'Piano', 'Saxophone', 'Shakers', 'Tambourine', 'Trombone', 'Trumpet', 'Ukulele', 'vibraphone', 'Violin']


In [2]:
def get_mixes_array(data_dir):
    data = []
    path = os.path.join(data_dir+"full_mix")
    for img in os.listdir(data_dir+"full_mix"):
        for stem in os.listdir(data_dir+"stems"):
            if img.split("_")[0] == stem.split("_")[0]:
                temp=stem.split("_",1)
                label = temp[1].split(".")[0]
                class_num = labels.index(label)
                try:
                    img_arr = cv2.imread(os.path.join(path,img),0)
                    resized_arr = cv2.resize(img_arr, (img_size, img_size))
                    data.append([resized_arr,class_num])
                    gc.collect()
                except Exception as e:
                    print(e)
    return np.array(data,dtype="object") 


In [3]:
def get_stems_array(data_dir):
    data = []
    path = os.path.join(data_dir)
    for img in os.listdir(data_dir):
            try:
                img_arr = cv2.imread(os.path.join(path,img),0)
                resized_arr = cv2.resize(img_arr, (img_size, img_size))
                data.append([resized_arr])
                gc.collect()
            except Exception as e:
                print(e)
    return np.array(data,dtype="float32") 

In [4]:
x_train = get_mixes_array("music_separation_2/train/")

y_train = get_stems_array("music_separation_2/train/stems")

#x_test = get_mixes_array("Music_separation_dataset/test/")

#y_test = get_stems_array("Music_separation_dataset/test/stems")

x_valid = get_mixes_array("music_separation_2/valid/")

y_valid = get_stems_array("music_separation_2/valid/stems")



In [5]:
x_train_mix=[]
x_train_label=[]

#x_test_mix=[]
#x_test_label=[]

x_valid_mix=[]
x_valid_label=[]

for feature, label in x_train:
    x_train_mix.append(feature)
    x_train_label.append(label)

#for feature, label in x_test:
    #x_test_mix.append(feature)
    #x_test_label.append(label)

for feature, label in x_valid:
    x_valid_mix.append(feature)
    x_valid_label.append(label)

del x_train
#del x_test
del x_valid

In [6]:
gc.collect()
x_train_mix = np.array(x_train_mix)/255
gc.collect()
#x_test_mix = np.array(x_test_mix)/255
#gc.collect()
x_valid_mix = np.array(x_valid_mix)/255
gc.collect()
y_train = np.array(y_train)/255
gc.collect()
#y_test = np.array(y_test)/255
#gc.collect()
y_valid = np.array(y_valid)/255
gc.collect()


0

In [7]:
x_train_mix = x_train_mix.reshape(-1, img_size, img_size, 1)
y_train = y_train.reshape(-1, img_size, img_size, 1)

x_valid_mix = x_valid_mix.reshape(-1, img_size, img_size, 1)
y_valid = y_valid.reshape(-1, img_size, img_size, 1)

#x_test_mix = x_test_mix.reshape(-1, img_size, img_size, 1)
#y_test = y_test.reshape(-1, img_size, img_size, 1)


In [8]:
x_train_label = np.array(x_train_label)
x_train_label = to_categorical(x_train_label)

#x_test_label = np.array(x_test_label)
#x_test_label = to_categorical(x_test_label)

x_valid_label = np.array(x_valid_label)
x_valid_label = to_categorical(x_valid_label)

In [9]:
print(x_valid_label.shape)
print(x_train_mix.shape)
print(x_train_label.shape)
print(y_train.shape)

(423, 24)
(2574, 128, 128, 1)
(2574, 24)
(2574, 128, 128, 1)


In [ ]:
num_classes = 24
def Unet():
    inputs =  Input(shape=(None,None,1))
    label_input = Input(shape=(num_classes,))

    conv1 = Conv2D(16, (3,3), activation = 'relu', padding='same')(inputs)
    conv1 = BatchNormalization()(conv1)
    conv1 = Conv2D(16, (3,3), activation = 'relu', padding='same')(conv1)
    conv1 = BatchNormalization()(conv1)
    pool1 = MaxPool2D((2,2))(conv1)

    conv2 = Conv2D(32, (3,3), activation = 'relu', padding='same')(pool1)
    conv2 = BatchNormalization()(conv2)
    conv2 = Conv2D(32, (3,3), activation = 'relu', padding='same')(conv2)
    conv2 = BatchNormalization()(conv2)
    pool2 = MaxPool2D((2,2))(conv2)

    conv3 = Conv2D(64, (3,3), activation = 'relu', padding='same')(pool2)
    conv3 = BatchNormalization()(conv3)
    conv3 = Conv2D(64, (3,3), activation = 'relu', padding='same')(conv3)
    conv3 = BatchNormalization()(conv3)
    pool3 = MaxPool2D((2,2))(conv3)

    conv4 = Conv2D(128, (3,3), activation = 'relu', padding='same')(pool3)
    conv4 = BatchNormalization()(conv4)
    conv4 = Conv2D(128, (3,3), activation = 'relu', padding='same')(conv4)
    conv4 = BatchNormalization()(conv4)
    pool4 = MaxPool2D((2,2))(conv4)

    conv5 = Conv2D(256, (3,3), activation = 'relu', padding='same')(pool4)
    conv5 = BatchNormalization()(conv5)
    conv5 = Conv2D(256, (3,3), activation = 'relu', padding='same')(conv5)
    conv5 = BatchNormalization()(conv5)
    
    label_lay = Dense(256,activation="relu")(label_input)
    label_lay = Reshape((1,1,256))(label_lay)
    label_lay = UpSampling2D((8,8))(label_lay)
    multi_bottle = concatenate([conv5,label_lay])
    
    goUp1 = Conv2DTranspose(128,(3,3),activation = 'relu',padding="same",strides=(2,2))(multi_bottle)
    goUp1 = BatchNormalization()(goUp1)
    goUp1 = concatenate([goUp1,conv4])
    conv6 = Conv2D(128, (3,3), activation = 'relu', padding='same')(goUp1)
    conv6 = BatchNormalization()(conv6)
    conv6 = Conv2D(128, (3,3), activation = 'relu', padding='same')(conv6)
    conv6 = BatchNormalization()(conv6)

    goUp2 = Conv2DTranspose(64,(3,3),activation = 'relu',padding="same",strides=(2,2))(conv6)
    goUp2 = BatchNormalization()(goUp2)
    goUp2 = concatenate([goUp2,conv3])
    conv7 = Conv2D(64, (3,3), activation = 'relu', padding='same')(goUp2)
    conv7 = BatchNormalization()(conv7)
    conv7 = Conv2D(64, (3,3), activation = 'relu', padding='same')(conv7)
    conv7 = BatchNormalization()(conv7)

    goUp3 = Conv2DTranspose(32,(3,3),activation = 'relu',padding="same",strides=(2,2))(conv7)
    goUp3 = BatchNormalization()(goUp3)
    goUp3 = concatenate([goUp3,conv2])
    conv8 = Conv2D(32, (3,3), activation = 'relu', padding='same')(goUp3)
    conv8 = BatchNormalization()(conv8)
    conv8 = Conv2D(32, (3,3), activation = 'relu', padding='same',)(conv8)
    conv8 = BatchNormalization()(conv8)

    goUp4 = Conv2DTranspose(16,(3,3),activation = 'relu',padding="same",strides=(2,2))(conv8)
    goUp4 = BatchNormalization()(goUp4)
    goUp4 = concatenate([goUp4,conv1])
    conv9 = Conv2D(16, (3,3), activation = 'relu', padding='same')(goUp4)
    conv9 = BatchNormalization()(conv9)
    conv9 = Conv2D(16, (3,3), activation = 'relu', padding='same')(conv9)
    conv9 = BatchNormalization()(conv9)


    outputs = Conv2D(1,(1,1),activation="relu")(conv9)

    model = Model(inputs=[inputs,label_input],outputs=[outputs])
    return model

model = Unet()
model.compile(
              optimizer = 'adam', loss = 'mae',
              metrics = ['mae']
              )
     

In [25]:
model.summary()

Model: "functional_4"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_8       │ (None, None,      │          0 │ -                 │
│ (InputLayer)        │ None, 1)          │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_80 (Conv2D)  │ (None, None,      │        160 │ input_layer_8[0]… │
│                     │ None, 16)         │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, None,      │         64 │ conv2d_80[0][0]   │
│ (BatchNormalizatio… │ None, 16)         │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_81 (Conv2D)  │ (None, None,      │      2,320 │ batch_normalizat… │
│                     │ None, 16)         │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, None,      │         64 │ conv2d_81[0][0]   │
│ (BatchNormalizatio… │ None, 16)         │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_16    │ (None, None,      │          0 │ batch_normalizat… │
│ (MaxPooling2D)      │ None, 16)         │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_82 (Conv2D)  │ (None, None,      │      4,640 │ max_pooling2d_16… │
│                     │ None, 32)         │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, None,      │        128 │ conv2d_82[0][0]   │
│ (BatchNormalizatio… │ None, 32)         │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_83 (Conv2D)  │ (None, None,      │      9,248 │ batch_normalizat… │
│                     │ None, 32)         │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, None,      │        128 │ conv2d_83[0][0]   │
│ (BatchNormalizatio… │ None, 32)         │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_17    │ (None, None,      │          0 │ batch_normalizat… │
│ (MaxPooling2D)      │ None, 32)         │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_84 (Conv2D)  │ (None, None,      │     18,496 │ max_pooling2d_17… │
│                     │ None, 64)         │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, None,      │        256 │ conv2d_84[0][0]   │
│ (BatchNormalizatio… │ None, 64)         │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_85 (Conv2D)  │ (None, None,      │     36,928 │ batch_normalizat… │
│                     │ None, 64)         │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, None,      │        256 │ conv2d_85[0][0]   │
│ (BatchNormalizatio… │ None, 64)         │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_18    │ (None, None,      │          0 │ batch_normalizat… │
│ (MaxPooling2D)      │ None, 64)         │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_86 (Conv2D)  │ (None, None,      │     73,856 │ max_pooling2d_18

 Total params: 2,466,577 (9.41 MB)

 Trainable params: 2,463,153 (9.40 MB)

 Non-trainable params: 3,424 (13.38 KB)

In [17]:
checkpoint = ModelCheckpoint('Checkpoint.keras',"val_loss", save_freq=4500,mode="min")

In [26]:
batch_size = 8
n_epochs = 400
model.fit(x=[x_train_mix,x_train_label], y=y_train, batch_size = batch_size,
                    epochs = n_epochs, validation_data = ([x_valid_mix,x_valid_label], y_valid),callbacks=[checkpoint],shuffle=True)

Epoch 1/400
322/322 ━━━━━━━━━━━━━━━━━━━━ 75s 207ms/step - loss: 0.1536 - mae: 0.1536 - val_loss: 0.1922 - val_mae: 0.1922
Epoch 2/400
322/322 ━━━━━━━━━━━━━━━━━━━━ 66s 205ms/step - loss: 0.1270 - mae: 0.1270 - val_loss: 0.1333 - val_mae: 0.1333
Epoch 3/400
322/322 ━━━━━━━━━━━━━━━━━━━━ 66s 206ms/step - loss: 0.1240 - mae: 0.1240 - val_loss: 0.1848 - val_mae: 0.1848
Epoch 4/400
322/322 ━━━━━━━━━━━━━━━━━━━━ 64s 200ms/step - loss: 0.1243 - mae: 0.1243 - val_loss: 0.1217 - val_mae: 0.1217
Epoch 5/400
322/322 ━━━━━━━━━━━━━━━━━━━━ 63s 195ms/step - loss: 0.1219 - mae: 0.1219 - val_loss: 0.1263 - val_mae: 0.1263
Epoch 6/400
322/322 ━━━━━━━━━━━━━━━━━━━━ 61s 188ms/step - loss: 0.1192 - mae: 0.1192 - val_loss: 0.1185 - val_mae: 0.1185
Epoch 7/400
322/322 ━━━━━━━━━━━━━━━━━━━━ 60s 188ms/step - loss: 0.1182 - mae: 0.1182 - val_loss: 0.1223 - val_mae: 0.1223
Epoch 8/400
322/322 ━━━━━━━━━━━━━━━━━━━━ 60s 187ms/step - loss: 0.1179 - mae: 0.1179 - val_loss: 0.1221 - val_mae: 0.1221
Epoch 9/400
322/322 ━━━━

In [37]:
gc.collect()

model.save('my_separator_model_2.keras')

In [27]:
img = cv2.imread("Music_separation_dataset/train/mix/0_mix.png",0)

img=cv2.resize(img,(128,128))
img= np.reshape(img,(-1, 128, 128, 1))
img=img/255
drum = np.array([[0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0,0,0,0,0,0,0,0,0,0,0,0]])

predimg= np.squeeze(model.predict([img,drum]))

prediction= predimg*255
print(prediction.shape)

cv2.imwrite("separated_drum_part2.png",prediction)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 376ms/step
(128, 128)


True

In [29]:
import librosa

image = cv2.imread("separated_drum_part2.png",0)

to_db = (image.astype(np.float32)/255)*80-80

power = librosa.db_to_power(to_db)

audio = librosa.feature.inverse.mel_to_audio(power,n_fft = 2048, hop_length = 512,n_iter=256,sr=22050)


audio = librosa.util.normalize(audio)

import soundfile
soundfile.write('drum_prediction.wav',audio, 22050)




In [30]:
img = cv2.imread("Music_separation_dataset/train/mix/462_mix.png",0)

img=cv2.resize(img,(128,128))
img= np.reshape(img,(-1, 128, 128, 1))
img=img/255
acoustic = np.array([[0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,0,0,0,0,0,0,0,0,0,0,0]])

predimg= np.squeeze(model.predict([img,acoustic]))

prediction= predimg*255
print(prediction.shape)

cv2.imwrite("separated_part_acoustic2.png",prediction)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 41ms/step
(128, 128)


True

In [31]:
import librosa

image = cv2.imread("separated_part_acoustic2.png",0)

to_db = (image.astype(np.float32)/255)*80-80

power = librosa.db_to_power(to_db)

audio = librosa.feature.inverse.mel_to_audio(power,n_fft = 2048, hop_length = 512,n_iter=256,sr=22050)


audio = librosa.util.normalize(audio)

import soundfile
soundfile.write('acoustic_prediction2.wav',audio, 22050)

In [32]:
img = cv2.imread("Music_separation_dataset/train/mix/19_mix.png",0)

img=cv2.resize(img,(128,128))
img= np.reshape(img,(-1, 128, 128, 1))
img=img/255
acoustic = np.array([[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,0,0,0,0,0,0,0,1,0,0,0]])

predimg= np.squeeze(model.predict([img,acoustic]))

prediction= predimg*255
print(prediction.shape)

cv2.imwrite("separated_part_Trumpet2.png",prediction)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 43ms/step
(128, 128)


True

In [33]:
import librosa

image = cv2.imread("separated_part_Trumpet2.png",0)

to_db = (image.astype(np.float32)/255)*80-80

power = librosa.db_to_power(to_db)

audio = librosa.feature.inverse.mel_to_audio(power,n_fft = 2048, hop_length = 512,n_iter=256,sr=22050)


audio = librosa.util.normalize(audio)

import soundfile
soundfile.write('trumpet_prediction2.wav',audio, 22050)

In [34]:
freq, sr = librosa.load("audio_separator_dataset/mix/0.wav")

image = cv2.imread("separated_drum_part2.png",0)

image = cv2.resize(image,(130,128))

to_db = (image.astype(np.float32)/255)*80-80

power = librosa.db_to_power(to_db)

short_time= librosa.stft(freq)

mag,phase = librosa.magphase(short_time)

inverse_mel = librosa.feature.inverse.mel_to_stft(power,sr=sr)



short_inverse = inverse_mel * np.exp(phase)

recon = librosa.istft(short_inverse)
recon = librosa.util.normalize(recon)

soundfile.write('drum_pediction_Phase2.wav',recon, 22050)


In [35]:
freq, sr = librosa.load("audio_separator_dataset/mix/462.wav")

image = cv2.imread("separated_part_acoustic2.png",0)

image = cv2.resize(image,(130,128))

to_db = (image.astype(np.float32)/255)*80-80

power = librosa.db_to_power(to_db)

short_time= librosa.stft(freq)

mag,phase = librosa.magphase(short_time)

inverse_mel = librosa.feature.inverse.mel_to_stft(power,sr=sr)



short_inverse = inverse_mel * np.exp(1j*phase)

recon = librosa.istft(short_inverse)
recon = librosa.util.normalize(recon)

soundfile.write('acoustic_pediction_Phase2.wav',recon, 22050)

In [36]:
freq, sr = librosa.load("audio_separator_dataset/mix/19.wav")

image = cv2.imread("separated_part_trumpet2.png",0)

image = cv2.resize(image,(130,128))

to_db = (image.astype(np.float32)/255)*80-80

power = librosa.db_to_power(to_db)

short_time= librosa.stft(freq)

mag,phase = librosa.magphase(short_time)

inverse_mel = librosa.feature.inverse.mel_to_stft(power,sr=sr)



short_inverse = inverse_mel * np.exp(1j*phase)

recon = librosa.istft(short_inverse)
recon = librosa.util.normalize(recon)

soundfile.write('trumpet_pediction_Phase2.wav',recon, 22050)

In [38]:
img = cv2.imread("Music_separation_dataset/train/mix/0_mix.png",0)

img=cv2.resize(img,(128,128))
img= np.reshape(img,(-1, 128, 128, 1))
img=img/255
acoustic = np.array([[1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,0,0,0,0,0,0,0,0,0,0,0]])

predimg= np.squeeze(model.predict([img,acoustic]))

prediction= predimg*255
print(prediction.shape)

cv2.imwrite("separated_part_accodrian.png",prediction)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 40ms/step
(128, 128)


True

In [40]:
image = cv2.imread("separated_part_accodrian.png",0)

to_db = (image.astype(np.float32)/255)*80-80

power = librosa.db_to_power(to_db)

audio = librosa.feature.inverse.mel_to_audio(power,n_fft = 2048, hop_length = 512,n_iter=256,sr=22050)


audio = librosa.util.normalize(audio)


soundfile.write('separated_part_accordian.wav',audio, 22050)

In [43]:
img = cv2.imread("Music_separation_dataset/train/mix/2_mix.png",0)

img=cv2.resize(img,(128,128))
img= np.reshape(img,(-1, 128, 128, 1))
img=img/255
acoustic = np.array([[0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,0,0,0,0,0,0,0,0,0,0,0]])

predimg= np.squeeze(model.predict([img,acoustic]))

prediction= predimg*255
print(prediction.shape)

cv2.imwrite("separated_part_banjo.png",prediction)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 42ms/step
(128, 128)


True

In [44]:
image = cv2.imread("separated_part_banjo.png",0)

to_db = (image.astype(np.float32)/255)*80-80

power = librosa.db_to_power(to_db)

audio = librosa.feature.inverse.mel_to_audio(power,n_fft = 2048, hop_length = 512,n_iter=256,sr=22050)


audio = librosa.util.normalize(audio)


soundfile.write('separated_part_banjo.wav',audio, 22050)

In [45]:
img = cv2.imread("Music_separation_dataset/train/mix/3_mix.png",0)

img=cv2.resize(img,(128,128))
img= np.reshape(img,(-1, 128, 128, 1))
img=img/255
acoustic = np.array([[0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0,0,0,0,0,0,0,0,0,0,0,0]])

predimg= np.squeeze(model.predict([img,acoustic]))

prediction= predimg*255
print(prediction.shape)

cv2.imwrite("separated_part_bass.png",prediction)


image = cv2.imread("separated_part_bass.png",0)

to_db = (image.astype(np.float32)/255)*80-80

power = librosa.db_to_power(to_db)

audio = librosa.feature.inverse.mel_to_audio(power,n_fft = 2048, hop_length = 512,n_iter=256,sr=22050)


audio = librosa.util.normalize(audio)


soundfile.write('separated_part_bass.wav',audio, 22050)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 43ms/step
(128, 128)


In [46]:
img = cv2.imread("Music_separation_dataset/train/mix/4_mix.png",0)

img=cv2.resize(img,(128,128))
img= np.reshape(img,(-1, 128, 128, 1))
img=img/255
acoustic = np.array([[0, 0, 0, 0,1, 0, 0, 0, 0, 0, 0, 0, 0,0,0,0,0,0,0,0,0,0,0,0]])

predimg= np.squeeze(model.predict([img,acoustic]))

prediction= predimg*255
print(prediction.shape)

cv2.imwrite("separated_part_clarinet.png",prediction)


image = cv2.imread("separated_part_clarinet.png",0)

to_db = (image.astype(np.float32)/255)*80-80

power = librosa.db_to_power(to_db)

audio = librosa.feature.inverse.mel_to_audio(power,n_fft = 2048, hop_length = 512,n_iter=256,sr=22050)


audio = librosa.util.normalize(audio)


soundfile.write('separated_part_clarinet.wav',audio, 22050)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 40ms/step
(128, 128)


In [55]:
img = cv2.imread("Music_separation_dataset/train/mix/5_mix.png",0)

img=cv2.resize(img,(128,128))
img= np.reshape(img,(-1, 128, 128, 1))
img=img/255
acoustic = np.array([[0, 0, 0, 0,0, 1, 0, 0, 0, 0, 0, 0, 0,0,0,0,0,0,0,0,0,0,0,0]])

predimg= np.squeeze(model.predict([img,acoustic]))

prediction= predimg*255
print(prediction.shape)

cv2.imwrite("separated_part_cowbell.png",prediction)


image = cv2.imread("separated_part_cowbell.png",0)

to_db = (image.astype(np.float32)/255)*80-80

power = librosa.db_to_power(to_db)

audio = librosa.feature.inverse.mel_to_audio(power,n_fft = 2048, hop_length = 512,n_iter=256,sr=22050)


audio = librosa.util.normalize(audio)


soundfile.write('separated_part_cowbell.wav',audio, 22050)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 36ms/step
(128, 128)


In [49]:
img = cv2.imread("Music_separation_dataset/train/mix/6_mix.png",0)

img=cv2.resize(img,(128,128))
img= np.reshape(img,(-1, 128, 128, 1))
img=img/255
acoustic = np.array([[0, 0, 0, 0,0, 1, 0, 0, 0, 0, 0, 0, 0,0,0,0,0,0,0,0,0,0,0,0]])

predimg= np.squeeze(model.predict([img,acoustic]))

prediction= predimg*255
print(prediction.shape)

cv2.imwrite("separated_part_dobro.png",prediction)


image = cv2.imread("separated_part_dobro.png",0)

to_db = (image.astype(np.float32)/255)*80-80

power = librosa.db_to_power(to_db)

audio = librosa.feature.inverse.mel_to_audio(power,n_fft = 2048, hop_length = 512,n_iter=256,sr=22050)


audio = librosa.util.normalize(audio)


soundfile.write('separated_part_dobro.wav',audio, 22050)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 36ms/step
(128, 128)


In [51]:
img = cv2.imread("Music_separation_dataset/train/mix/7_mix.png",0)

img=cv2.resize(img,(128,128))
img= np.reshape(img,(-1, 128, 128, 1))
img=img/255
acoustic = np.array([[0, 0, 0, 0,0, 0, 1, 0, 0, 0, 0, 0, 0,0,0,0,0,0,0,0,0,0,0,0]])

predimg= np.squeeze(model.predict([img,acoustic]))

prediction= predimg*255
print(prediction.shape)

cv2.imwrite("separated_part_eletric.png",prediction)


image = cv2.imread("separated_part_eletric.png",0)

to_db = (image.astype(np.float32)/255)*80-80

power = librosa.db_to_power(to_db)

audio = librosa.feature.inverse.mel_to_audio(power,n_fft = 2048, hop_length = 512,n_iter=256,sr=22050)


audio = librosa.util.normalize(audio)


soundfile.write('separated_part_eletric.wav',audio, 22050)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 38ms/step
(128, 128)


In [53]:
img = cv2.imread("Music_separation_dataset/train/mix/8_mix.png",0)

img=cv2.resize(img,(128,128))
img= np.reshape(img,(-1, 128, 128, 1))
img=img/255
acoustic = np.array([[0, 0, 0, 0,0, 0, 0,0, 1, 0, 0, 0, 0,0,0,0,0,0,0,0,0,0,0,0]])

predimg= np.squeeze(model.predict([img,acoustic]))

prediction= predimg*255
print(prediction.shape)

cv2.imwrite("separated_part_flute.png",prediction)


image = cv2.imread("separated_part_flute.png",0)

to_db = (image.astype(np.float32)/255)*80-80

power = librosa.db_to_power(to_db)

audio = librosa.feature.inverse.mel_to_audio(power,n_fft = 2048, hop_length = 512,n_iter=256,sr=22050)


audio = librosa.util.normalize(audio)


soundfile.write('separated_part_flute.wav',audio, 22050)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 42ms/step
(128, 128)


In [58]:
img = cv2.imread("Music_separation_dataset/train/mix/17_mix.png",0)

img=cv2.resize(img,(128,128))
img= np.reshape(img,(-1, 128, 128, 1))
img=img/255
acoustic = np.array([[0, 0, 0, 0,0, 0, 0,0, 0, 0, 0, 0, 0,0,0,0,0,0,1,0,0,0,0,0]])

predimg= np.squeeze(model.predict([img,acoustic]))

prediction= predimg*255
print(prediction.shape)

cv2.imwrite("separated_part_tambourine.png",prediction)


image = cv2.imread("separated_part_tambourine.png",0)

to_db = (image.astype(np.float32)/255)*80-80

power = librosa.db_to_power(to_db)

audio = librosa.feature.inverse.mel_to_audio(power,n_fft = 2048, hop_length = 512,n_iter=256,sr=22050)


audio = librosa.util.normalize(audio)


soundfile.write('separated_part_tambourine.wav',audio, 22050)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step
(128, 128)


In [60]:
img = cv2.imread("Music_separation_dataset/train/stems/17_Tambourine.png",0)

img=cv2.resize(img,(128,128))
img= np.reshape(img,(-1, 128, 128, 1))
img=img/255
acoustic = np.array([[0, 0, 0, 0,0, 0, 0,0, 0, 0, 0, 0, 0,0,0,0,0,0,1,0,0,0,0,0]])

predimg= np.squeeze(model.predict([img,acoustic]))

prediction= predimg*255
print(prediction.shape)

cv2.imwrite("separated_part_tambourine.png",prediction)


image = cv2.imread("separated_part_tambourine2.png",0)

to_db = (image.astype(np.float32)/255)*80-80

power = librosa.db_to_power(to_db)

audio = librosa.feature.inverse.mel_to_audio(power,n_fft = 2048, hop_length = 512,n_iter=256,sr=22050)


audio = librosa.util.normalize(audio)


soundfile.write('separated_part_tambourine2.wav',audio, 22050)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 42ms/step
(128, 128)


AttributeError: 'NoneType' object has no attribute 'astype'